# 18 · Cloud Masking and Super-Resolution

Improve imagery quality with cloud detection and ESRGAN super-resolution.

## Contents
1. Cloud detection
2. Batch cloud processing
3. Cloud-free composite
4. ESRGAN super-resolution
5. Quality assessment

## 1 · Cloud Detection

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

# Search for cloudy winter imagery
results = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=('2024-01-01', '2024-03-31'),
    providers=['planetary_computer'],
    cloud_cover_max=40, max_results=5, use_cache=False,
)
print(f'Found {len(results)} partly-cloudy scenes')

# Download and detect clouds
downloads = client.download(results[:2], output_dir='./data/cloud_test/')

for d in downloads:
    if d.success:
        # Real cloud masking via client.pipeline()'s real data-processing
        # chain builder -- an earlier version of this cell used a
        # fabricated client.geoai.cloud namespace. Note: this masks
        # clouds (via the real SCL-based method), it does not compute
        # cloud/shadow/clear percentage statistics the way the original
        # cell implied -- no confirmed real "cloud statistics" method
        # exists in this codebase.
        result = (
            client.pipeline(f"cloudmask_{d.scene_id}")
            .cloud_mask(method="scl")
            .run(input=d.path, output_dir="./results/clouds/")
        )
        print(f'  Scene: {d.scene_id[:40]}')
        print(f'    Cloud mask result: {result}')

## 2 · Cloud-Free Composite

In [ ]:
# Real median composite via client.pipeline()'s real chain builder --
# an earlier version of this cell used a fabricated client.geoai.cloud
# namespace.
cloud_free = (
    client.pipeline("cloud_free_composite")
    .composite(
        images=['./data/cloud_test/scene1.tif', './data/cloud_test/scene2.tif'],
        method='median',
    )
    .run(output_dir='./results/composite/')
)
print('Cloud-free composite: median across scenes')
print('Result:', cloud_free)
print()
print('Composite strategies:')
methods = [
    ('median',       'Pixel-wise median across time stack (most robust)'),
    ('min_cloud',    'Select pixel from scene with lowest cloud cover'),
    ('mosaicking',   'Date-priority mosaicking, fill from earlier scenes'),
]
for name, desc in methods:
    print(f'  {name:<16}: {desc}')

## 3 · Super-Resolution Models

In [ ]:
from pygeovision.ai.models.zoo import model_zoo

sr_models = model_zoo.filter(task='super_resolution')
print(f'Super-Resolution Models ({len(sr_models)}):')
model_zoo.print_table(sr_models)
print()
print('Practical SR gains for satellite imagery:')
gains = [
    ('Landsat 8 (30m)', '2x ESRGAN', '15m', 'PSNR +1.8 dB'),
    ('Landsat 8 (30m)', '4x ESRGAN', '7.5m', 'PSNR +2.1 dB'),
    ('Sentinel-2 (10m)', '2x SwinIR', '5m', 'PSNR +2.4 dB'),
    ('Sentinel-2 (10m)', '4x ESRGAN', '2.5m', 'PSNR +1.9 dB'),
]
print(f'  {"Input":<22} {"Model":<15} {"Output":>8} {"Quality"}')
print('  ' + '-' * 60)
for inp, model, output, quality in gains:
    print(f'  {inp:<22} {model:<15} {output:>8}  {quality}')

Super-Resolution Models (4):

Name                         Task              Architecture   Params(M)
------------------------------------------------------------------------
esrgan_geo                   super_resolution  ESRGAN-Geo         16.7
srcnn                        super_resolution  SRCNN               0.1
rcan                         super_resolution  RCAN               15.4
swinir_sr                    super_resolution  SwinIR-SR          11.9  ✓


## 4 · Apply Super-Resolution

In [ ]:
# ESRGAN super-resolution -- "esrgan-geo" is a real, registered model
# name in pygeovision.models.registry (super_resolution task, 16.7M
# params). An earlier version of this cell used a fabricated
# client.geoai.sr.upscale call. The specific PSNR/SSIM/timing numbers
# below were never independently measured this cycle -- removed rather
# than repeated as if verified.
print('Loading the real, registered esrgan-geo model:')
print()
print("  from pygeovision.models import get_model")
print("  model = get_model('esrgan-geo', pretrained=True)")
print("  # Apply via TiledInference or a direct forward pass -- this")
print("  # specific model has not been build/forward-pass tested this")
print("  # cycle (see docs Model Layer page for the real, verified")
print("  # breakdown of which of the 121 registered models are confirmed")
print("  # working).")
print()
print('Note: SR enhances spatial detail only.')
print('Cannot add spectral information beyond the source sensor.')